<h1 style="text-align:center;"><b>Proyecto 3 - Othello</b></h1>
<h3 style="text-align:center;">Marcos Díaz (221102), Daniel Machic (22118), Maria Jose Ramírez (221051)</h3>

**GitHub**: https://github.com/MarcosDiaz1409/Proyecto3-IA.git


## **Importar Librerías**

In [ ]:
import numpy as np
import pandas as pd
import pygame
import time
import math
import random
from copy import deepcopy

## **Representación del tablero**

In [ ]:
EMPTY=0
BLACK=1
WHITE=-1

## **Clase OthelloEngine**
Esta clase `OthelloEngine` constituye el núcleo lógico del juego. Es responsable de gestionar el estado del tablero, validar movimientos, aplicar las reglas de Othello, realizar el volteo de fichas y determinar cuándo una partida ha finalizado. Esta separación permite reutilizar la lógica del juego independientemente de la interfaz gráfica o del agente inteligente utilizado.

In [ ]:
class OthelloEngine:

    EMPTY = 0
    BLACK = 1
    WHITE = -1

    DIRECTIONS = [
        (-1, -1), (-1, 0), (-1, 1),
        (0, -1),           (0, 1),
        (1, -1),  (1, 0),  (1, 1)
    ]

    def __init__(self):

        self.board = np.zeros((8, 8), dtype=int)

        # Posición inicial oficial
        self.board[3][3] = self.WHITE
        self.board[3][4] = self.BLACK
        self.board[4][3] = self.BLACK
        self.board[4][4] = self.WHITE

        self.current_player = self.BLACK

    # ==================================================
    # UTILIDADES
    def is_on_board(self, row, col):
        return 0 <= row < 8 and 0 <= col < 8

    def get_opponent(self, player):
        return -player

    def clone(self):

        new_game = OthelloEngine()

        new_game.board = self.board.copy()
        new_game.current_player = self.current_player

        return new_game

    # ==================================================
    # MOVIMIENTOS
    def get_flips(self, row, col, player):

        if not self.is_on_board(row, col):
            return []

        if self.board[row][col] != self.EMPTY:
            return []

        opponent = self.get_opponent(player)

        flips = []

        for dr, dc in self.DIRECTIONS:

            r = row + dr
            c = col + dc

            temp_flips = []

            while (
                self.is_on_board(r, c)
                and self.board[r][c] == opponent
            ):

                temp_flips.append((r, c))

                r += dr
                c += dc

            if (
                len(temp_flips) > 0
                and self.is_on_board(r, c)
                and self.board[r][c] == player
            ):
                flips.extend(temp_flips)

        return flips

    def is_valid_move(self, row, col, player):

        return len(self.get_flips(row, col, player)) > 0

    def get_legal_moves(self, player):

        legal_moves = []

        for row in range(8):
            for col in range(8):

                if self.is_valid_move(row, col, player):
                    legal_moves.append((row, col))

        return legal_moves

    def has_valid_move(self, player):

        return len(self.get_legal_moves(player)) > 0

    def apply_move(self, row, col, player):

        flips = self.get_flips(row, col, player)

        if len(flips) == 0:
            return False

        self.board[row][col] = player

        for r, c in flips:
            self.board[r][c] = player

        self.current_player = self.get_opponent(player)

        return True

    def pass_turn(self):

        self.current_player = self.get_opponent(
            self.current_player
        )

    # ==================================================
    # ESTADO DEL JUEGO
    def count_discs(self):

        black_count = np.sum(
            self.board == self.BLACK
        )

        white_count = np.sum(
            self.board == self.WHITE
        )

        return black_count, white_count

    def is_board_full(self):

        return not np.any(
            self.board == self.EMPTY
        )

    def is_terminal(self):

        return (
            self.is_board_full()
            or
            (
                not self.has_valid_move(self.BLACK)
                and
                not self.has_valid_move(self.WHITE)
            )
        )

    def get_winner(self):

        black_count, white_count = self.count_discs()

        if black_count > white_count:
            return self.BLACK

        elif white_count > black_count:
            return self.WHITE

        return 0

    # ==================================================
    # DEBUG / VISUALIZACIÓN
    def print_board(self):

        symbols = {
            self.EMPTY: ".",
            self.BLACK: "B",
            self.WHITE: "W"
        }

        print("  0 1 2 3 4 5 6 7")

        for r in range(8):

            row_string = f"{r} "

            row_string += " ".join(
                symbols[cell]
                for cell in self.board[r]
            )

            print(row_string)

    def print_score(self):

        black_count, white_count = self.count_discs()

        print(
            f"Black: {black_count} | "
            f"White: {white_count}"
        )